# Data Cleaning and Similarity Feature Vectors


This notebook combines Spotify audio descriptors with Last.fm popularity statistics and listener tags to build reproducible feature spaces for similarity search and clustering.


# Imports and constants


In [92]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime
import json
import re
import unicodedata
from typing import Dict, List

import numpy as np
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 120)

cwd = Path.cwd().resolve()
NOTEBOOK_DIR = (
    cwd
    if cwd.name == "1-data-cleaning-and-feature-vector"
    else cwd / "1-data-cleaning-and-feature-vector"
)
if not NOTEBOOK_DIR.exists():
    raise FileNotFoundError(f"Couldn't locate notebook directory at {NOTEBOOK_DIR}")
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_ROOT
ARTIFACT_DIR = NOTEBOOK_DIR / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
AUDIO_BASE_COLS = [
    "acousticness",
    "danceability",
    "energy",
    "instrumentalness",
    "liveness",
    "loudness",
    "speechiness",
    "valence",
    "tempo",
]
POPULARITY_VALUE_COLS = ["lfm_playcount", "lfm_listeners"]
POPULARITY_FEATURES = [
    "log_playcount",
    "log_listeners",
    "log_plays_per_listener",
]
TAG_TEXT_COL = "tags_clean_text"
MAX_TAGS_PER_TRACK = 10

# Helper Functions for Normalization, Tag Cleaning, and Weighting


In [93]:
REMOVE_BRACKETS_RE = re.compile(r"\([^)]*\)|\[[^\]]*\]")
NON_ALNUM_RE = re.compile(r"[^a-z0-9]+")
SPACE_RE = re.compile(r"\s+")
TAG_CHAR_FILTER_RE = re.compile(r"[^a-z0-9#+&/-]+")

TAG_STOPLIST = {
    "album",
    "albums",
    "single",
    "singles",
    "soundtrack",
    "downloads",
    "download",
    "seen live",
    "live",
    "english",
    "spanish",
    "japanese",
    "instrumental",
    "music",
    "rock music",
    "pop music",
    "the best",
    "best",
    "favorites",
    "favorite",
    "favourite",
    "under 2000 listeners",
    "airplay",
}
TAG_CANONICAL_MAP = {
    "hip hop": "hip-hop",
    "hiphop": "hip-hop",
    "hip-hop": "hip-hop",
    "r&b": "rnb",
    "rnb": "rnb",
    "r and b": "rnb",
    "indie rock": "indie-rock",
    "alt rock": "alternative rock",
    "indie pop": "indie-pop",
    "lo fi": "lo-fi",
    "lofi": "lo-fi",
    "electro pop": "electropop",
}


def strip_accents(text: str) -> str:
    return "".join(
        ch
        for ch in unicodedata.normalize("NFKD", text)
        if not unicodedata.combining(ch)
    )


def normalize_text(value: str) -> str:
    if not isinstance(value, str):
        return ""
    text = value.lower().strip()
    text = REMOVE_BRACKETS_RE.sub(" ", text)
    text = strip_accents(text)
    text = NON_ALNUM_RE.sub(" ", text)
    return SPACE_RE.sub(" ", text).strip()


def canonicalize_tag(tag: str) -> str:
    tag = strip_accents(tag.lower())
    tag = tag.replace("&", " and ")
    tag = tag.replace("/", " ")
    tag = TAG_CHAR_FILTER_RE.sub(" ", tag)
    tag = SPACE_RE.sub(" ", tag).strip()
    if not tag:
        return ""
    tag = TAG_CANONICAL_MAP.get(tag, tag)
    if tag.endswith("s") and len(tag) > 4:
        singular = tag[:-1]
        if singular not in TAG_STOPLIST:
            tag = singular
    return tag


def clean_tag_blob(blob: str, max_tags: int = MAX_TAGS_PER_TRACK) -> List[str]:
    if not isinstance(blob, str):
        return []
    raw_tags = [token.strip() for token in blob.split(";") if token.strip()]
    cleaned: List[str] = []
    for raw in raw_tags:
        tag = canonicalize_tag(raw)
        if not tag:
            continue
        if tag in TAG_STOPLIST:
            continue
        if tag in cleaned:
            continue
        cleaned.append(tag)
        if len(cleaned) >= max_tags:
            break
    return cleaned


def winsorize_series(series: pd.Series, upper_q: float = 0.995) -> pd.Series:
    if series.dropna().empty:
        return series
    cap = series.quantile(upper_q)
    return series.clip(lower=0, upper=cap)


def apply_block_weights(
    matrix: np.ndarray, dims: Dict[str, int], weights: Dict[str, float]
) -> np.ndarray:
    weighted = np.asarray(matrix).copy()
    start = 0
    for block in ["audio", "popularity", "tags"]:
        dim = dims.get(block, 0)
        weight = weights.get(block, 1.0)
        if dim <= 0:
            continue
        stop = start + dim
        weighted[:, start:stop] *= weight
        start = stop
    return weighted

# Loading, Combining, and Cleaning


In [94]:
# Load inputs, build join keys, and deduplicate tracks
spotify_df = pd.read_csv(DATA_DIR / "dataset_clean.csv")
lastfm_df = pd.read_csv(DATA_DIR / "dataset_with_lastfm_all.csv")
lastfm_df = lastfm_df.drop(columns=["Unnamed: 0"], errors="ignore")

for frame in (spotify_df, lastfm_df):
    frame["norm_track"] = frame["track_name"].map(normalize_text)
    frame["norm_artist"] = frame["artists"].map(normalize_text)

search_lookup = spotify_df[
    ["norm_track", "norm_artist", "search_string"]
].drop_duplicates(subset=["norm_track", "norm_artist"], keep="first")
tracks = lastfm_df.merge(
    search_lookup,
    on=["norm_track", "norm_artist"],
    how="left",
)
print(
    f"Joined table: {len(tracks):,} rows; {tracks['track_id'].nunique():,} distinct track IDs"
)

genre_signatures = pd.read_csv(DATA_DIR / "genre_avg_signatures.csv")
genre_defaults = {
    col: genre_signatures.set_index("track_genre")[col].to_dict()
    for col in AUDIO_BASE_COLS
}
for col, mapping in genre_defaults.items():
    tracks[col] = tracks[col].fillna(tracks["track_genre"].map(mapping))

completeness_cols = AUDIO_BASE_COLS + POPULARITY_VALUE_COLS + ["lfm_duration_ms"]
tracks["missing_ratio"] = tracks[completeness_cols].isna().mean(axis=1)
before = len(tracks)
tracks = (
    tracks.sort_values(by=["missing_ratio", "popularity"], ascending=[True, False])
    .drop_duplicates(subset=["norm_artist", "norm_track"], keep="first")
    .drop(columns=["missing_ratio"])
    .reset_index(drop=True)
)
print(f"Deduplicated to {len(tracks):,} rows (dropped {before - len(tracks):,})")

tracks = tracks[tracks["track_id"].notna()].copy()
tracks["track_id"] = tracks["track_id"].astype(str)
print(f"Remaining unique tracks: {tracks['track_id'].nunique():,}")

print()

# Clean Last.fm popularity signals and align durations
for col in POPULARITY_VALUE_COLS + ["lfm_duration_ms"]:
    tracks[col] = pd.to_numeric(tracks[col], errors="coerce")
for col in POPULARITY_VALUE_COLS:
    tracks.loc[tracks[col] < 0, col] = np.nan
    tracks[col] = winsorize_series(tracks[col], upper_q=0.995)

plays_per_listener = tracks["lfm_playcount"] / tracks["lfm_listeners"].replace(
    {0: np.nan}
)
tracks["log_playcount"] = np.log1p(tracks["lfm_playcount"])
tracks["log_listeners"] = np.log1p(tracks["lfm_listeners"])
tracks["log_plays_per_listener"] = np.log1p(plays_per_listener)
tracks["plays_per_listener"] = plays_per_listener

valid_duration_mask = (
    tracks["lfm_duration_ms"].notna()
    & tracks["duration_ms"].notna()
    & (tracks["duration_ms"] > 0)
)
duration_ratio = (
    tracks.loc[valid_duration_mask, "lfm_duration_ms"]
    / tracks.loc[valid_duration_mask, "duration_ms"]
)
bad_ratio_idx = duration_ratio[(duration_ratio < 0.5) | (duration_ratio > 2.0)].index
print(
    f"Dropping {len(bad_ratio_idx):,} rows with inconsistent Spotify/Last.fm duration"
)
tracks = tracks.drop(index=bad_ratio_idx).reset_index(drop=True)

tracks = tracks[tracks["duration_ms"] > 0]
tracks = tracks[tracks["tempo"] > 0]
tracks["tempo_log"] = np.log(tracks["tempo"].clip(lower=1.0))
tracks["duration_minutes"] = tracks["duration_ms"] / 60000.0
print(
    f"After sanity checks: {len(tracks):,} rows, {tracks['track_id'].nunique():,} unique track IDs"
)

print()

# Clean and canonicalize Last.fm tags
tracks["tags_clean_list"] = tracks["lfm_tags"].apply(clean_tag_blob)
tracks["tags_clean_text"] = tracks["tags_clean_list"].apply(lambda tags: " ".join(tags))
tracks["tag_count"] = tracks["tags_clean_list"].apply(len)
tag_coverage = (tracks["tag_count"] > 0).mean()
print(f"Tag coverage: {tag_coverage:.1%} of tracks retain >=1 cleaned tag")

Joined table: 73,608 rows; 73,608 distinct track IDs
Deduplicated to 70,172 rows (dropped 3,436)
Remaining unique tracks: 70,172

Dropping 14,961 rows with inconsistent Spotify/Last.fm duration
After sanity checks: 55,129 rows, 55,129 unique track IDs

Tag coverage: 92.2% of tracks retain >=1 cleaned tag


# Cleaned, Combined, and Transformed Dataset for Modeling


In [95]:
# Assembling modeling table and create train/validation
# #TODO: TRAIN/VALIDATION MIGHT NOT BE NECESSARY
modeling_cols = list(
    dict.fromkeys(
        [
            "track_id",
            "artists",
            "track_name",
            "track_genre",
            "search_string",
            "duration_ms",
            "tempo",
            "tempo_log",
            "duration_minutes",
            TAG_TEXT_COL,
        ]
        + AUDIO_BASE_COLS
        + POPULARITY_VALUE_COLS
        + POPULARITY_FEATURES
    )
)
modeling_df = tracks[modeling_cols + ["plays_per_listener", "tag_count"]].copy()
modeling_df[TAG_TEXT_COL] = modeling_df[TAG_TEXT_COL].fillna("")
modeling_df["track_genre"] = modeling_df["track_genre"].fillna("unknown")
genre_counts = modeling_df["track_genre"].value_counts()
rare_mask = modeling_df["track_genre"].map(genre_counts) < 2
split_labels = modeling_df["track_genre"].where(~rare_mask, other="other-rare")
train_df, val_df, split_train, split_val = train_test_split(
    modeling_df,
    split_labels,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=split_labels,
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
assert not train_df["track_id"].duplicated().any()
assert not val_df["track_id"].duplicated().any()
print(
    f"Train: {len(train_df):,} tracks | Validation: {len(val_df):,} tracks | Stratify labels: {split_labels.nunique()}"
)

print()

# Exporting cleaned modeling dataset as csv file
clean_dataset_path = ARTIFACT_DIR / "full_dataset_clean.csv"
modeling_df.to_csv(clean_dataset_path, index=False)
size_mb = clean_dataset_path.stat().st_size / (1024 * 1024)
print(
    f"Saved cleaned dataset → {clean_dataset_path.relative_to(NOTEBOOK_DIR)} ({size_mb:.1f} MB)"
)

print()

# Combined matrix
print(
    f"Combined feature matrix: {modeling_df.shape[0]:,} tracks × {modeling_df.shape[1]-1} features"
)

# Displays part of the dataset
modeling_df.head()

Train: 44,103 tracks | Validation: 11,026 tracks | Stratify labels: 113

Saved cleaned dataset → artifacts/full_dataset_clean.csv (19.4 MB)

Combined feature matrix: 55,129 tracks × 24 features


,track_id,artists,track_name,track_genre,search_string,duration_ms,tempo,tempo_log,duration_minutes,tags_clean_text,acousticness,danceability,energy,instrumentalness,liveness,loudness,speechiness,valence,lfm_playcount,lfm_listeners,log_playcount,log_listeners,log_plays_per_listener,plays_per_listener,tag_count
0,3nqQXoyQOWXiESFLlDF1hG,Sam Smith;Kim Petras,Unholy (feat. Kim Petras),dance,"Unholy (feat. Kim Petras) - Sam Smith, Kim Petras",156943,131.121,4.876121,2.615717,pop hyperpop electropop electronic pop rap,0.01300,0.714,0.472,0.000005,0.2660,-7.375,0.0864,0.238,10995250.0,920570.0,16.212974,13.732749,2.560629,11.943959,5
1,4uUG5RXrOk84mYEfFvj3cK,David Guetta;Bebe Rexha,I'm Good (Blue),dance,"I'm Good (Blue) - David Guetta, Bebe Rexha",175238,128.040,4.852343,2.920633,house electronic dance electro house 2022,0.00383,0.561,0.965,0.000007,0.3710,-3.673,0.0343,0.304,7924612.0,758980.0,15.885484,13.539732,2.437215,10.441134,5
2,5ww2BF9slyYgNOk37BlC4u,Manuel Turizo,La Bachata,latin,La Bachata - Manuel Turizo,162637,124.980,4.828154,2.710617,reggaeton latin pop latin pop latino bachata male vocalist colombian pop colombia colombian,0.58300,0.835,0.679,0.000002,0.2180,-5.329,0.0364,0.850,4325531.0,365331.0,15.280046,12.808562,2.552568,11.840033,10
3,6Sq7ltF9Qa7SNFBsV5Cogx,Bad Bunny;Chencho Corleone,Me Porto Bonito,latin,"Me Porto Bonito - Bad Bunny, Chencho Corleone",178567,92.005,4.521843,2.976117,bad bunny reggaeton fire latin chencho corleone,0.09010,0.911,0.712,0.000027,0.0933,-5.105,0.0817,0.425,8928528.0,632558.0,16.004762,13.357529,2.715685,14.114955,5
4,5Eax0qFko2dh7Rl2lYs3bx,Bad Bunny,Efecto,latin,Efecto - Bad Bunny,213061,98.047,4.585447,3.551017,reggaeton latin,0.14100,0.801,0.475,0.000017,0.0639,-8.797,0.0516,0.234,6119925.0,420681.0,15.627061,12.949632,2.743910,14.547662,2


# Building Feature Pipeline (audio + popularity + tags)


In [96]:
AUDIO_FEATURES = AUDIO_BASE_COLS + ["tempo_log", "duration_minutes"]
tag_pipe = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                min_df=5,
                max_df=0.5,
                ngram_range=(1, 2),
                max_features=5000,
            ),
        ),
        ("svd", TruncatedSVD(n_components=100, random_state=RANDOM_STATE)),
        ("scale", StandardScaler()),
    ]
)
audio_pipe = Pipeline(
    steps=[
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("pca", PCA(n_components=0.95, random_state=RANDOM_STATE)),
    ]
)
pop_pipe = Pipeline(
    steps=[
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]
)
feature_pipeline = ColumnTransformer(
    transformers=[
        ("audio", audio_pipe, AUDIO_FEATURES),
        ("popularity", pop_pipe, POPULARITY_FEATURES),
        ("tags", tag_pipe, TAG_TEXT_COL),
    ],
    remainder="drop",
    sparse_threshold=0.0,
)
feature_pipeline.fit(train_df)
train_features = feature_pipeline.transform(train_df)
val_features = feature_pipeline.transform(val_df)
audio_dim = (
    feature_pipeline.named_transformers_["audio"].named_steps["pca"].n_components_
)
tag_dim = feature_pipeline.named_transformers_["tags"].named_steps["svd"].n_components
pop_dim = len(POPULARITY_FEATURES)
block_dims = {"audio": audio_dim, "popularity": pop_dim, "tags": tag_dim}
print(
    f"Block dims → audio PCA: {audio_dim}, popularity: {pop_dim}, tag SVD: {tag_dim}; total = {train_features.shape[1]}"
)
pca_cumvar = (
    feature_pipeline.named_transformers_["audio"]
    .named_steps["pca"]
    .explained_variance_ratio_.cumsum()
)
print(
    "Audio PCA cumulative variance (first 5 components):",
    np.round(pca_cumvar[:5], 3),
)

Block dims → audio PCA: 8, popularity: 3, tag SVD: 100; total = 111
Audio PCA cumulative variance (first 5 components): [0.282 0.45  0.58  0.696 0.779]


# Apply Block Weights and Persist Artifacts


In [97]:
BLOCK_WEIGHTS = {"audio": 1.0, "popularity": 0.4, "tags": 0.9}
weighted_train = apply_block_weights(train_features, block_dims, BLOCK_WEIGHTS)
weighted_val = apply_block_weights(val_features, block_dims, BLOCK_WEIGHTS)
# timestamp = datetime.utcnow().replace(microsecond=0).isoformat() + "Z"
feature_cols = [f"feature_{i:03d}" for i in range(weighted_train.shape[1])]

# Aggregate feature vectors
all_track_ids = np.concatenate(
    [
        train_df["track_id"].to_numpy(),
        val_df["track_id"].to_numpy(),
    ]
)
all_features = np.vstack([train_features, val_features])
all_weighted = np.vstack([weighted_train, weighted_val])
full_df = pd.concat([train_df, val_df], ignore_index=True)

all_npz_path = ARTIFACT_DIR / "features_apt_all.npz"
np.savez_compressed(
    all_npz_path,
    track_ids=all_track_ids,
    features=all_features,
    features_weighted=all_weighted,
)

all_csv_path = ARTIFACT_DIR / "features_apt_all_weighted.csv"
all_features_df = pd.DataFrame(all_weighted, columns=feature_cols)
all_features_df.insert(0, "track_id", all_track_ids)
all_features_df.to_csv(all_csv_path, index=False)

print(
    f"Combined feature matrix → {all_features.shape[0]:,} tracks × {all_features.shape[1]} dims"
)

print()

print(f"Saved {all_npz_path.relative_to(NOTEBOOK_DIR)}")
print(f"Saved {all_csv_path.relative_to(NOTEBOOK_DIR)}")

vectorizer_payload = {
    "pipeline": feature_pipeline,
    "audio_features": AUDIO_FEATURES,
    "popularity_features": POPULARITY_FEATURES,
    "tag_column": TAG_TEXT_COL,
    "block_dims": block_dims,
    "block_weights": BLOCK_WEIGHTS,
}
vectorizer_path = ARTIFACT_DIR / "vectorizers.joblib"
joblib.dump(vectorizer_payload, vectorizer_path)

tfidf_params = (
    feature_pipeline.named_transformers_["tags"].named_steps["tfidf"].get_params()
)
block_dims_json = {key: int(value) for key, value in block_dims.items()}
config = {
    # "created_at_utc": timestamp,
    "random_state": RANDOM_STATE,
    "records": {"train": len(train_df), "validation": len(val_df)},
    "block_dims": block_dims_json,
    "block_weights": BLOCK_WEIGHTS,
    "columns": {
        "audio": AUDIO_FEATURES,
        "popularity": POPULARITY_FEATURES,
        "tag_text": TAG_TEXT_COL,
    },
    "tag_vectorizer": {
        "min_df": tfidf_params["min_df"],
        "max_df": tfidf_params["max_df"],
        "ngram_range": tfidf_params["ngram_range"],
        "max_features": tfidf_params["max_features"],
        "svd_components": int(block_dims["tags"]),
    },
    "artifacts": {
        # "train_features": str(train_npz.relative_to(NOTEBOOK_DIR)),
        # "val_features": str(val_npz.relative_to(NOTEBOOK_DIR)),
        "all_features": str(all_npz_path.relative_to(NOTEBOOK_DIR)),
        "vectorizers": str(vectorizer_path.relative_to(NOTEBOOK_DIR)),
    },
}
config_path = ARTIFACT_DIR / "features_config.json"
with config_path.open("w", encoding="utf-8") as fp:
    json.dump(config, fp, indent=2)

print("Saved artifacts:")
for path in [all_npz_path, vectorizer_path, config_path]:
    size_kb = path.stat().st_size / 1024
    print(f"- {path.relative_to(NOTEBOOK_DIR)} ({size_kb:,.1f} KB)")

# Inspect PCA loadings
all_features_df.head()

Combined feature matrix → 55,129 tracks × 111 dims

Saved artifacts/features_apt_all.npz
Saved artifacts/features_apt_all_weighted.csv
Saved artifacts:
- artifacts/features_apt_all.npz (85,976.8 KB)
- artifacts/vectorizers.joblib (4,118.8 KB)
- artifacts/features_config.json (0.9 KB)


,track_id,feature_000,feature_001,feature_002,feature_003,feature_004,feature_005,feature_006,feature_007,feature_008,feature_009,feature_010,feature_011,feature_012,feature_013,...,feature_096,feature_097,feature_098,feature_099,feature_100,feature_101,feature_102,feature_103,feature_104,feature_105,feature_106,feature_107,feature_108,feature_109,feature_110
0,4ckoZlzQTOTEN8rhfQPbBl,2.065026,0.871409,-0.340064,-0.566777,-1.063053,0.409325,-1.313289,-1.048944,-0.092122,-0.070957,-0.167369,-0.606534,-0.011121,0.263575,...,-0.873634,-0.446184,-0.935219,-1.353143,0.260243,-0.595473,-2.488638,0.097101,-0.438052,0.666867,0.317411,0.605472,-0.219065,-0.301099,-0.510811
1,3CRtbOoAP82y0VbPO5j1ft,-1.605333,0.389619,0.904004,0.268017,-0.286908,-1.006378,-0.814541,-0.586951,0.486121,0.445361,0.468991,0.265444,-0.776643,0.091603,...,-0.622018,-0.355343,-0.043709,-0.272950,1.039185,0.314686,-1.827511,0.299801,-0.872529,-0.033383,-0.136339,-0.023530,1.038657,-0.149268,-1.080481
2,265SFiKBPvo5ojatV1egRk,1.604891,-1.149363,0.168046,0.712593,0.418015,1.377352,-1.027740,-1.114899,0.330919,0.302715,0.310755,-0.287993,-0.237761,1.932151,...,0.488059,-0.510801,0.055894,0.479026,-0.462355,0.126475,1.085610,-0.169211,-0.351221,0.833178,-0.101159,-0.688632,-0.217280,0.314536,-0.389322
3,7oQ8pTAe783QS0UwOmbHRb,0.790360,-1.851698,-0.299230,0.321908,0.294828,1.121776,-1.337076,0.011193,-0.460269,-0.468981,-0.232099,-0.727250,-0.000721,-0.147608,...,0.028169,0.027375,-0.019588,0.422632,-1.494018,-0.323037,-0.037075,-1.391343,0.589210,-0.554779,0.405575,-0.274338,1.103543,-0.318250,-1.501014
4,46Rgoqf0TIsHPudbaaqUP0,-1.826262,-2.470739,1.263439,-0.369585,-0.594776,-0.576660,0.214143,0.557400,-0.417733,-0.392050,-0.358194,-0.240648,-0.359403,-0.064857,...,0.462145,0.748940,-0.752569,-0.451734,-0.252735,0.680069,0.058906,0.579301,-0.265201,-0.443728,0.015983,-0.100816,-0.141127,0.140316,0.240047


In [98]:
audio_pca = feature_pipeline.named_transformers_["audio"].named_steps["pca"]
audio_components = pd.DataFrame(
    audio_pca.components_,
    columns=AUDIO_FEATURES,
    index=[f"PC{i+1}" for i in range(audio_pca.n_components_)],
)
audio_components["explained_variance_ratio"] = audio_pca.explained_variance_ratio_
print("Audio PCA components (loadings per raw audio feature):")
audio_components.head()

loadings_path = ARTIFACT_DIR / "audio_pca_loadings.csv"
audio_components.to_csv(loadings_path)
print(f"Saved PCA loadings → {loadings_path.relative_to(NOTEBOOK_DIR)}")

audio_components.head()

Audio PCA components (loadings per raw audio feature):
Saved PCA loadings → artifacts/audio_pca_loadings.csv


,acousticness,danceability,energy,instrumentalness,liveness,loudness,speechiness,valence,tempo,tempo_log,duration_minutes,explained_variance_ratio
PC1,-0.410278,0.187957,0.472163,-0.260166,0.083873,0.474396,0.062493,0.256160,0.313101,0.325695,0.000746,0.281728
PC2,-0.017980,-0.370784,-0.055902,0.301286,-0.077318,-0.156752,-0.188342,-0.320720,0.541523,0.528698,0.163232,0.168113
PC3,0.282187,0.396100,-0.323348,-0.144057,-0.401925,-0.144415,-0.218195,0.432520,0.247782,0.258609,-0.304651,0.130101
PC4,0.330563,-0.092599,-0.104625,-0.175298,0.552277,-0.137644,0.584335,0.046249,0.211474,0.188666,-0.304335,0.115866
PC5,0.097414,0.402774,-0.103979,0.108587,0.053498,-0.145080,0.299746,0.168578,0.058976,0.076646,0.808796,0.082840


**Interpretation**

- The combined feature matrix stacks every train and validation song so downstream consumers (LLMs, retrievers, or analytics jobs) can work from one canonical table of block-weighted similarity vectors per `track_id`.
- The PCA loadings describe how raw audio descriptors co-vary: each component highlights dominant trait mixes (e.g., energy vs. acousticness). Inspecting the explained-variance column reveals how much overall sonic variation each component captures, helping decide how many dimensions to keep for future tasks.


# Targeted KNNs for tracks in weighted space


In [99]:
from sklearn.neighbors import NearestNeighbors

target_songs = [
    ("HUMBLE.", "Kendrick Lamar"),
    ("Landslide", "Fleetwood Mac"),
    ("Do I Wanna Know?", "Arctic Monkeys"),
    ("Un Ratito", "Bad Bunny"),
    ("Customer Is King", "Solomun"),
    ("Too Sad To Cry", "Sasha Alex Sloan"),
    ("Sound Of Wind Driven Rain", "Will Ackerman"),
]

knn = NearestNeighbors(metric="cosine", algorithm="brute")
knn.fit(all_weighted)

full_aug = full_df.assign(
    _norm_track=full_df["track_name"].map(normalize_text),
    _norm_artist=full_df["artists"].map(normalize_text),
)

for title, artist in target_songs:
    exact_mask = (
        full_df["track_name"].str.lower().fillna("") == title.lower()
    ) & full_df["artists"].str.lower().fillna("").str.contains(artist.lower())
    if not exact_mask.any():
        norm_mask = (full_aug["_norm_track"] == normalize_text(title)) & (
            full_aug["_norm_artist"] == normalize_text(artist)
        )
    else:
        norm_mask = exact_mask

    if not norm_mask.any():
        print(f"Track not found in dataset: {title} — {artist}")
        print("-")
        continue

    idx = norm_mask[norm_mask].index[0]
    row = full_df.loc[idx]
    distances, indices = knn.kneighbors(all_weighted[idx : idx + 1], n_neighbors=6)
    print(f"Seed track: {row['track_name']} — {row['artists']}")
    for rank, neighbor_idx in enumerate(indices[0][1:], start=1):
        neighbor = full_df.iloc[neighbor_idx]
        print(
            f"  {rank}. {neighbor['track_name']} — {neighbor['artists']} (cosine {distances[0][rank]:.3f})"
        )
    print("-")

Seed track: HUMBLE. — Kendrick Lamar
  1. R.I.P. — Playboi Carti (cosine 0.026)
  2. SICKO MODE — Travis Scott (cosine 0.038)
  3. INDUSTRY BABY (feat. Jack Harlow) — Lil Nas X;Jack Harlow (cosine 0.039)
  4. Pass — Nizi19;Karamel19 (cosine 0.064)
  5. Best Friend — Young Thug (cosine 0.075)
-
Track not found in dataset: Landslide — Fleetwood Mac
-
Seed track: Do I Wanna Know? — Arctic Monkeys
  1. Mad Sounds — Arctic Monkeys (cosine 0.049)
  2. Cigarette Daydreams — Cage The Elephant (cosine 0.052)
  3. Afraid — The Neighbourhood (cosine 0.067)
  4. Is This It — The Strokes (cosine 0.068)
  5. Last Nite — The Strokes (cosine 0.075)
-
Seed track: Un Ratito — Bad Bunny
  1. MIA (feat. Drake) — Bad Bunny;Drake (cosine 0.002)
  2. Me Fui de Vacaciones — Bad Bunny (cosine 0.004)
  3. Yo Perreo Sola — Bad Bunny (cosine 0.011)
  4. Dos Mil 16 — Bad Bunny (cosine 0.014)
  5. Aguacero — Bad Bunny (cosine 0.019)
-
Seed track: Customer Is King — Solomun
  1. After Rain comes Sun — Solomun (cosin

# Random k-NN Sample in Weighted Space


In [100]:
from sklearn.neighbors import NearestNeighbors

sample_tracks = full_df.sample(3, random_state=RANDOM_STATE)["track_id"].tolist()
knn = NearestNeighbors(metric="cosine", algorithm="brute")
knn.fit(all_weighted)
id_to_idx = {tid: idx for idx, tid in enumerate(full_df["track_id"])}
for tid in sample_tracks:
    idx = id_to_idx[tid]
    distances, indices = knn.kneighbors(all_weighted[idx : idx + 1], n_neighbors=6)
    neighbors = indices[0][1:]
    print(
        f"Seed track: {full_df.loc[idx, 'track_name']} — {full_df.loc[idx, 'artists']}"
    )
    for rank, neighbor_idx in enumerate(neighbors, start=1):
        neighbor = full_df.iloc[neighbor_idx]
        print(
            f"  {rank}. {neighbor['track_name']} — {neighbor['artists']} (cosine {distances[0][rank]:.3f})"
        )
    print("-")

Seed track: For Her — MMOTHS;Young & Sick
  1. Emerging Space — Spuntic (cosine 0.293)
  2. Where the Waters Meet (O Waly, Waly) [Arr. C.B. Chambers for Wind Symphony] [Live] — Traditional;Mansfield Wind Symphony;Jeff King (cosine 0.298)
  3. Rain For Sleeping — Rain Sounds;Rain for Deep Sleep;BodyHI (cosine 0.306)
  4. Lucia di Lammermoor - Spargi d'amaro pianto - Alt. Version — Gaetano Donizetti;Vincenzo Ciliberti (cosine 0.320)
  5. Spejlbilledet — Pernille Højgaard (cosine 0.325)
-
Seed track: Cien kilos de barro (Remastered) — Enrique Guzman
  1. Pensaba En Ti - En Vivo — Enrique Guzman (cosine 0.021)
  2. Con y por Amor — Enrique Guzman (cosine 0.022)
  3. Quiero Ser Libre/Anoche No Dormí - En Vivo — Enrique Guzman (cosine 0.029)
  4. Payasito (Ponchinello) — Enrique Guzman (cosine 0.030)
  5. Ángel de Mi Vida - Angel of the Morning — Enrique Guzman (cosine 0.030)
-
Seed track: Chain of Life — 2002
  1. The Emerald Way — 2002 (cosine 0.016)
  2. River of Stars — 2002 (cosine 0.08

In [101]:
from sklearn.neighbors import NearestNeighbors

sample_tracks = train_df.sample(3, random_state=RANDOM_STATE)["track_id"].tolist()
knn = NearestNeighbors(metric="cosine", algorithm="brute")
knn.fit(weighted_train)
id_to_idx = {tid: idx for idx, tid in enumerate(train_df["track_id"])}
for tid in sample_tracks:
    idx = id_to_idx[tid]
    distances, indices = knn.kneighbors(weighted_train[idx : idx + 1], n_neighbors=6)
    neighbors = indices[0][1:]
    print(
        f"Seed track: {train_df.loc[idx, 'track_name']} — {train_df.loc[idx, 'artists']}"
    )
    for rank, neighbor_idx in enumerate(neighbors, start=1):
        neighbor = train_df.iloc[neighbor_idx]
        print(
            f"  {rank}. {neighbor['track_name']} — {neighbor['artists']} (cosine {distances[0][rank]:.3f})"
        )
    print("-")

Seed track: Beatnik Beach — The Go-Go's
  1. This Old Feeling — The Go-Go's (cosine 0.020)
  2. Teenline — The Shivvers (cosine 0.211)
  3. One Way or Another — Blondie (cosine 0.223)
  4. I Think We're Alone Now — The Rubinoos (cosine 0.240)
  5. Heaven Is a Place On Earth — Belinda Carlisle (cosine 0.245)
-
Seed track: Good as Hell — Lizzo
  1. About Damn Time — Lizzo (cosine 0.018)
  2. Special — Lizzo (cosine 0.021)
  3. Boys - Pink Panda Remix — Lizzo;Pink Panda (cosine 0.044)
  4. Coldplay — Lizzo (cosine 0.077)
  5. Truth Hurts — Lizzo (cosine 0.089)
-
Seed track: You Make It Worse — Ouse
  1. Too Many Problems — Ouse;Powfu (cosine 0.025)
  2. Lovemark — Ouse;Powfu (cosine 0.040)
  3. Tô Só Observando — DJ Jamaika (cosine 0.094)
  4. Welcome To Los Santos — Oh No (cosine 0.124)
  5. Wesh Enfoiré #3 — Lesram (cosine 0.135)
-


## Ad-hoc similarity search helper


In [102]:
from sklearn.neighbors import NearestNeighbors

knn = NearestNeighbors(metric="cosine", algorithm="brute")
knn.fit(all_weighted)

full_aug = full_df.assign(
    _norm_track=full_df["track_name"].map(normalize_text),
    _norm_artist=full_df["artists"].map(normalize_text),
)


def _match_rows(df, title: str, artist: str | None):
    title_mask = df["track_name"].str.lower().fillna("") == title.lower()
    if artist:
        artist_mask = df["artists"].str.lower().fillna("").str.contains(artist.lower())
    else:
        artist_mask = True
    return title_mask & artist_mask


def find_similar_songs(title: str, artist: str | None = None, k: int = 10):
    mask = _match_rows(full_df, title, artist)
    if not mask.any():
        alt = full_aug.rename(
            columns={"_norm_track": "track_name", "_norm_artist": "artists"}
        )
        mask = _match_rows(
            alt, normalize_text(title), normalize_text(artist) if artist else None
        )

    if not mask.any():
        print(f"Track not found: {title} — {artist or 'any artist'}")
        return

    idx = mask[mask].index[0]
    row = full_df.loc[idx]
    distances, indices = knn.kneighbors(all_weighted[idx : idx + 1], n_neighbors=k + 1)
    print(f"Seed track: {row['track_name']} — {row['artists']}")
    for rank, neighbor_idx in enumerate(indices[0][1:], start=1):
        neighbor = full_df.iloc[neighbor_idx]
        print(
            f"  {rank}. {neighbor['track_name']} — {neighbor['artists']} (cosine {distances[0][rank]:.3f})"
        )

# Find similar songs using track and artist
find_similar_songs("HUMBLE.", "Kendrick Lamar")


Seed track: HUMBLE. — Kendrick Lamar
  1. R.I.P. — Playboi Carti (cosine 0.026)
  2. SICKO MODE — Travis Scott (cosine 0.038)
  3. INDUSTRY BABY (feat. Jack Harlow) — Lil Nas X;Jack Harlow (cosine 0.039)
  4. Pass — Nizi19;Karamel19 (cosine 0.064)
  5. Best Friend — Young Thug (cosine 0.075)
  6. BUTTERFLY EFFECT — Travis Scott (cosine 0.085)
  7. Look At Me! — XXXTENTACION (cosine 0.101)
  8. Trap Queen — Fetty Wap (cosine 0.102)
  9. RECON — Justin Stone (cosine 0.113)
  10. N95 — Kendrick Lamar (cosine 0.113)
